# hadeeth.id — bge-m3 vector build (Colab T4)

Builds the offline semantic-search artifact for the full **153,920-document** hadith corpus.

**Before running:** `Runtime` → `Change runtime type` → Hardware accelerator **T4 GPU** → Save.
Then `Runtime` → `Run all`.

## What it does
1. Guards on CUDA and installs `sentence-transformers`.
2. Fetches `vectorize_hadeeth.py` from R2, **pinned by sha256**, and runs it.
3. Verifies the artifact (row/id count, finiteness, L2 norms).
4. Zips and downloads it.

## Output — three files
| File | Format |
|---|---|
| `hadeeth_vectors_f16.bin` | `(153920, 1024)` raw float16, row-major, L2-normalised |
| `hadeeth_ids.ndjson` | one document id per line, same row order |
| `meta.json` | model, dims, per-collection counts, timing |

The corpus streams from `https://data.hadeeth.id` (920 objects, 1.65 GB) and the model
from HuggingFace. Expect roughly **40–70 minutes**: a counting pass, then the GPU embedding pass.
Keep the tab open; the runtime disconnects if idle too long.

Note: `vectors/` is a dtype-agnostic artifact. Cloudflare Vectorize stores float32, so a later
upload must upcast. Vectorize's free plan allows 5M stored vector dimensions against this
corpus's 157.6M, so bulk upload needs Workers Paid.

In [ ]:
# 1. GPU guard + dependencies
import subprocess, sys, torch

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > T4 GPU, then Run all.")
print("device:", torch.cuda.get_device_name(0))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
import sentence_transformers
print("sentence-transformers", sentence_transformers.__version__)


In [ ]:
# 2. Fetch the pinned vectorizer, then build the vectors
import hashlib, subprocess, sys, urllib.request

SCRIPT_SHA = "8111c8489209635d66f18f6bd16ce076fb8e57b7f78b6cf1b2e1059c7c625662"
URL = "https://pub-075ac3f4a31e4e85a876c26c815ae93d.r2.dev/scripts/vectorize_hadeeth.py"

req = urllib.request.Request(URL, headers={"User-Agent": "colab-hadeeth/1.0"})
body = urllib.request.urlopen(req, timeout=180).read()
got = hashlib.sha256(body).hexdigest()
assert got == SCRIPT_SHA, f"script hash mismatch\n  want {SCRIPT_SHA}\n  got  {got}"
open("vectorize_hadeeth.py", "wb").write(body)
print("script pinned ok:", got, len(body), "bytes")

subprocess.run([sys.executable, "vectorize_hadeeth.py"], check=True)


In [ ]:
# 3. Verify the artifact before downloading
import json, os
import numpy as np

VEC, IDS, DIMS = "hadeeth_vectors_f16.bin", "hadeeth_ids.ndjson", 1024

ids = [json.loads(l) for l in open(IDS, encoding="utf-8") if l.strip()]
rows = os.path.getsize(VEC) // (DIMS * 2)
assert os.path.getsize(VEC) % (DIMS * 2) == 0, "vector file is not a whole number of rows"
assert rows == len(ids), f"rows {rows} != ids {len(ids)}"
assert len(set(ids)) == len(ids), "duplicate ids"
assert all(len(i.encode()) <= 64 for i in ids), "an id exceeds the 64-byte Vectorize cap"

mm = np.memmap(VEC, dtype=np.float16, mode="r", shape=(rows, DIMS))
sample = np.asarray(mm[:: max(1, rows // 5000)][:5000], dtype=np.float32)
norms = np.linalg.norm(sample, axis=1)
assert np.isfinite(sample).all(), "non-finite values"
assert 0.97 <= norms.mean() <= 1.03, f"rows not L2-normalised (mean {norms.mean():.4f})"

print(f"rows           {rows:,}")
print(f"ids            {len(ids):,}  (unique, <=64 bytes)")
print(f"dtype          float16  dims {DIMS}")
print(f"finite         True")
print(f"L2 norm        mean {norms.mean():.5f}  min {norms.min():.5f}  max {norms.max():.5f}")
print(f"first id       {ids[0]}")
print(f"last id        {ids[-1]}")
print("VERIFIED\n")
print(json.dumps(json.load(open("meta.json", encoding="utf-8")), indent=2)[:800])


In [ ]:
# 4. Zip (stored, byte-exact) and download
import os, zipfile
from google.colab import files

z = "hadeeth_vectors.zip"
with zipfile.ZipFile(z, "w", zipfile.ZIP_STORED) as zf:
    for f in ("hadeeth_vectors_f16.bin", "hadeeth_ids.ndjson", "meta.json"):
        zf.write(f)
        print(f"  + {f}  {os.path.getsize(f)/1e6:,.1f} MB")
print("zip", round(os.path.getsize(z) / 1e6, 1), "MB")
files.download(z)
